# Loan Default Prediction - Modelling
## Objective
Building and evaluating supervised machine learning models to predict loan default.
This notebook covers data preprocessing, feature engineering, model training, 
hyperparameter tuning, and performance evaluation.

Models compared:
- Logistic Regression (baseline)
- Random Forest
- XGBoost

Evaluation metrics:
- AUC-ROC: measures how well the model separates defaulters from non-defaulters
- PR-AUC: precision-recall AUC, more informative than ROC-AUC under heavy class imbalance
- Brier Score: measures probability calibration — how accurate the predicted probabilities are

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, brier_score_loss,
                             average_precision_score, roc_curve,
                             precision_recall_curve)
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Data/application_train.csv')
print(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Data loaded: 307,511 rows, 122 columns


## 1. Preprocessing
### 1.1 Drop High Missing Columns
Dropping 49 columns with >40% missing values as identified in EDA. 
EXT_SOURCE_1 is retained despite 56% missingness due to its known 
predictive power in credit risk modelling.

In [3]:
# Drop columns with >40% missing except EXT_SOURCE_1
missing_pct = df.isnull().sum() / len(df) * 100
cols_to_drop = missing_pct[(missing_pct > 40) & (missing_pct.index != 'EXT_SOURCE_1')].index.tolist()
df = df.drop(columns=cols_to_drop)

print(f"Columns dropped: {len(cols_to_drop)}")
print(f"Remaining shape: {df.shape}")

Columns dropped: 48
Remaining shape: (307511, 74)


### 1.2 Imputation
Numerical columns imputed with median, categorical columns with mode 
except OCCUPATION_TYPE which is imputed with 'Unknown'.

In [4]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Impute numerical with median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Impute categorical
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        if col == 'OCCUPATION_TYPE':
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")

Remaining missing values: 0
Shape: (307511, 74)
